## Загрузка библиотек

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import numpy as np
import pandas as pd
from tqdm import tqdm

## Загрузка модели и данных

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

batch_size = 256
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)


val_sample_size = 2048
val_loader_fixed = torch.utils.data.DataLoader(valset, batch_size=val_sample_size, shuffle=True)
fixed_val_inputs, fixed_val_labels = next(iter(val_loader_fixed))
fixed_val_inputs = fixed_val_inputs.to(device)
fixed_val_labels = fixed_val_labels.to(device)


model = resnet18(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

100%|███████████████████████████████████████████████████████████████| 170498071/170498071 [01:54<00:00, 1494633.50it/s]


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified


## Обучение модели

In [5]:
logs = {
    'step': [],
    'val_loss':[],
    'val_accuracy': [],
    'poison_fraction':[]
}

num_epochs = 20
global_step = 0

for epoch in range(num_epochs):
    model.train()
    loop = tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        

        if np.random.rand() < 0.05:
            poison_fraction = np.random.uniform(0.2, 0.8)
            num_to_poison = int(poison_fraction * inputs.size(0))
            
            indices_to_poison = torch.randperm(inputs.size(0))[:num_to_poison]
            random_labels = torch.randint(0, 10, (num_to_poison,)).to(device)
            labels[indices_to_poison] = random_labels
            actual_poison_fraction = num_to_poison / inputs.size(0)
        else:
            actual_poison_fraction = 0.0

        optimizer.zero_grad()
        train_outputs = model(inputs)
        train_loss = criterion(train_outputs, labels)
        train_loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(fixed_val_inputs)
            val_loss = criterion(val_outputs, fixed_val_labels)
            
            _, predicted = val_outputs.max(1)
            correct = predicted.eq(fixed_val_labels).sum().item()
            val_acc = correct / val_sample_size
            
        model.train()
        
        logs['step'].append(global_step)
        logs['val_loss'].append(val_loss.item())
        logs['val_accuracy'].append(val_acc)
        logs['poison_fraction'].append(actual_poison_fraction)
        
        global_step += 1
        loop.set_postfix(val_loss=val_loss.item(), val_acc=val_acc, poison=actual_poison_fraction)

df_logs = pd.DataFrame(logs)
df_logs.to_csv('resnet_cifar_val_poison_logs.csv', index=False)
print("Logs saved to resnet_cifar_val_poison_logs.csv")

Epoch 20/20: 100%|██████████████████████████| 196/196 [08:03<00:00,  2.47s/it, poison=0, val_acc=0.744, val_loss=0.772]

Logs saved to resnet_cifar_val_poison_logs.csv
